# CS50 Lecture 3 - Algorithms

## 3.1 Week 3: From Step-by-Step Instructions to Measuring Efficiency

By Week 3, the basic building blocks of a program — variables, conditionals, loops, functions — are taken for granted, much as Week 2 took Scratch-derived syntax for granted in C. The focus shifts again, this time from writing correct code to writing *efficient* code. An **algorithm**, as introduced in Week 0, is step-by-step instructions for solving a problem; sorting a collection of data, in the everyday sense, means rearranging it from smallest to largest, alphabetically, or by some other ordering rule, while searching means locating a particular piece of information within a collection.

This lecture's two central problems — search and sort — are described as **canonical**: algorithms most computer scientists know, and the kind of question a technical interviewer is likely to ask. Beyond rote familiarity with specific algorithms, the broader goal is to build mental models and methodologies for comparing different approaches to the same problem, and a vocabulary (**running time**, **Big O**) for describing how a real-world algorithm scales when translated into code that a computer executes.

## 3.2 The Attendance Problem: Three Algorithms for Counting

### 3.2.1 Counting by Ones

Counting a roomful of people one at a time — 1, 2, 3, 4, ... — takes one step per person: for n people, the algorithm takes n steps. Graphed against the number of people in the room (x-axis) versus the number of steps taken (y-axis), this produces a straight line with slope 1.

### 3.2.2 Counting by Twos

Counting in increments of two — 2, 4, 6, 8, ... — still takes a number of steps proportional to n (specifically n/2 steps), so it is still graphed as a straight line, just one with a shallower slope: twice as fast as counting by ones for any room size, but fundamentally the same kind of growth — linear.

### 3.2.3 Divide and Conquer: Pairwise Summation

A third algorithm: each of n people starts by holding the value 1. Each then pairs off with another person who is still standing, adds the two held values together, and one person of each pair sits down holding the sum while the other remains standing. Repeating this pairing-and-summing step among those still standing causes the number of people left standing to halve on every repetition, until a single person remains holding the running total — the count of the room.

Applied to a room of people, the count obtained this way (**141**) closely matched, but did not exactly equal, a count obtained by counting one person at a time aloud (approximately **160**) — the discrepancy attributed to small arithmetic errors accumulating during the manual one-at-a-time count rather than to a flaw in the pairing algorithm itself.

This pairing algorithm is far faster than either of the first two for large n: doubling the number of people requires only one additional pairing round; quadrupling it requires only two additional rounds. The relationship between problem size and number of steps is no longer a straight line but flattens out — the algorithm is recognizable, in hindsight, as the same **divide-and-conquer** idea behind the Week 0 phone-book search, later formalized in this lecture as **binary search**.

## 3.3 Searching

### 3.3.1 The Search Problem as a Black Box

A search algorithm can be modeled as a black box: it accepts some input and produces an output. For the search problem specifically, the input is a collection of values — e.g., money sealed behind a row of numbered lockers — and the output is a single boolean: true if a target value is present, false otherwise.

Locations are addressed using zero-indexing: 7 lockers are numbered 0 through 6, not 1 through 7. Hearing a reference to "location 6" therefore implies at least 7 total locations.

A real computer cannot perceive an entire array at a glance the way a human eye can scan a printed grid of numbers. Each value behind a closed locker door must be checked one at a time — open the door, read the value, close the door, move to the next — which is the operating assumption behind all of the algorithms that follow.

### 3.3.2 Linear Search

Given 7 lockers holding unsorted denominations of money, the simplest strategy for locating a specific bill (e.g., USD 50) is to check each locker in turn, left to right, stopping as soon as it is found. Demonstrated on lockers holding, in order, USD 20, USD 500, USD 10, USD 5, USD 100, USD 1, and USD 50, the target (USD 50) is not found until the very last locker is opened — the worst case for this approach, since the search must continue until either the value is found or every locker has been checked.

Whether the scan proceeds left-to-right or right-to-left (or in any other fixed, consistent order) makes no fundamental difference if nothing is known about how the values are arranged: each ordering finds the target on the first try only by luck. Jumping around in a random order does not help either, and additionally requires tracking which lockers have already been checked — extra memory that becomes significant as the collection grows from 7 lockers to 700.

#### 3.3.2.1 Linear Search: Success Even Without a Match

A worked example on a plain 15-element array of integers, `11 23 8 14 30 9 6 17 22 28 25 15 7 10 19`, illustrates both outcomes of linear search. Searching for the target `9`: the scan checks `11`, `23`, `8`, `14`, `30`, and finds `9` at the sixth position, stopping immediately. Searching the same array for `50` (not present): the scan must check all 15 elements — every one of `11, 23, 8, 14, 30, 9, 6, 17, 22, 28, 25, 15, 7, 10, 19` — before concluding the value is absent.

This second case clarifies what "failure" means for a search algorithm. Linear search reporting `50` as not found is not a failure of the algorithm — it did exactly what it was asked: exhaustively check every element and correctly conclude the target is absent only after ruling out every position. **Linear search succeeds whether or not the target is present**, so long as it returns the correct boolean answer; an actual failure would be an *incorrect* answer (e.g., reporting "not found" without having checked every element). This is the mechanism behind the worst case (`O(n)`, scanning everything to confirm absence or find a value at the very end) and the best case (`Ω(1)`, finding the target on the very first check) established later in this lecture.

### 3.3.3 Binary Search

Given the same 7 lockers, now sorted from smallest to largest (USD 1, USD 5, USD 10, USD 20, USD 50, USD 100, USD 500), the search proceeds far more efficiently by checking the middle locker first, then discarding the half of the remaining lockers that cannot possibly contain the target (based on whether the target is smaller or larger than the middle value), and repeating on the remaining half. This **divide-and-conquer** strategy is the same approach used for the phone-book lookup in Week 0 and for the pairwise-counting algorithm earlier in this lecture.

#### 3.3.3.1 Binary Search: Tracking Start, End, and Middle

Implementing binary search concretely requires tracking three values as the search progresses: a **start** index, an **end** index, and a **middle** index computed from them — `middle = (start + end) / 2` (integer division). Each comparison against the value at `middle` narrows the sub-array by moving one of the two boundaries: if the target is greater than the middle value, `start` becomes `middle + 1`; if the target is less, `end` becomes `middle - 1`.

Applied to the sorted 15-element array `6 7 8 9 10 11 14 15 17 19 22 23 25 28 30` — the same 15 values used in 3.3.2.1's linear-search example, now sorted — searching for `19`:

| Step | Start | End | Middle | Value at Middle | Outcome |
|---|---|---|---|---|---|
| 1 | 0 | 14 | 7 | 15 | 19 > 15 → start = 8 |
| 2 | 8 | 14 | 11 | 23 | 19 < 23 → end = 10 |
| 3 | 8 | 10 | 9 | 19 | found |

Three steps locate the target, consistent with `O(log n)` (`log₂15 ≈ 3.9`).

Searching the same array for `16` (not present) illustrates how the algorithm recognizes absence:

| Step | Start | End | Middle | Value at Middle | Outcome |
|---|---|---|---|---|---|
| 1 | 0 | 14 | 7 | 15 | 16 > 15 → start = 8 |
| 2 | 8 | 14 | 11 | 23 | 16 < 23 → end = 10 |
| 3 | 8 | 10 | 9 | 19 | 16 < 19 → end = 8 |
| 4 | 8 | 8 | 8 | 17 | 16 < 17 → end = 7 |
| 5 | 8 | 7 | — | — | **start > end** |


> When `start` exceeds `end`, the sub-array being searched has shrunk to size 0 — the two boundaries have "crossed." This is the operational form of the empty-array guard already stated in 3.4.2's pseudocode ("if there are no doors left, return false"), and it is distinct from `start == end`: a single remaining element (step 4 above) is still checked before being ruled out. Only once the boundaries cross can the algorithm conclude the target is absent.

## 3.4 Search Algorithms in Pseudocode

### 3.4.1 Linear Search Pseudocode

`Given a row of numbered doors, determine whether the value 50 is behind one of them, and return true or false accordingly`

In English:
```text
For each door from left to right:
    If 50 is behind the door:
        return true
return false
```
The final `return false` belongs outside and after the loop, at the same indentation level as the `for` — not nested inside an `else` branch attached to the `if`. A tempting but incorrect variant:
```text
For each door from left to right:
    If 50 is behind the door:
        return true
    Else:
        return false
```
> This version is wrong: if the very first door checked does not hold the target, the `else` branch returns `false` immediately, without ever examining the remaining doors. `return` ends a function's execution immediately, so the rest of the loop never runs. The correct version waits until every door has been checked unsuccessfully before returning `false`.

In array notation, with `n` total doors indexed `0` through `n-1`:
```text
For i from 0 to n-1:
    If 50 is behind doors[i]:
        return true
return false
```
The loop bound is `n-1`, not `n`: with zero-indexing and `n` total elements, the last valid index is `n-1`.

Pseudocode of this kind functions as a deliberate amalgam of English, C, and (later) Python — writing the logic out in pseudocode first makes translating it into any specific language more direct, especially as the underlying problem grows more complex.

### 3.4.2 Binary Search Pseudocode

In English:
```text
If 50 is behind the middle door:
    return true
Else if 50 is less than the number behind the middle door:
    search the left half
Else if 50 is greater than the number behind the middle door:
    search the right half
```
A guard for the empty case must be added at the top, since each recursive halving eventually leaves no doors left to search:
```text
If there are no doors:
    return false
If 50 is behind the middle door:
    return true
Else if 50 is less than the number behind the middle door:
    search the left half
Else if 50 is greater than the number behind the middle door:
    search the right half
```
"Search the left half" / "search the right half" means applying this same procedure again to a smaller subset of doors — a self-referential technique formalized later in the lecture as **recursion**.

In array notation:
```text
If 50 is behind doors[middle]:
    return true
Else if 50 < doors[middle]:
    search doors[0] through doors[middle-1]
Else if 50 > doors[middle]:
    search doors[middle+1] through doors[n-1]
```
The middle index is the number of remaining doors divided by 2, rounded down (integer division) — for 7 doors, `7 / 2 = 3.5`, rounded down to index `3`, the true middle of a 7-element, zero-indexed array. Both halves deliberately exclude index `middle` itself, since that element was already checked by the preceding `if`.

## 3.5 Running Time and Big-O Notation

### 3.5.1 What Running Time Measures

**Running time** is not a clock reading in real seconds, since that figure would depend on the specific machine, its load at the time, and incidental implementation details. Instead, an algorithm's running time is expressed as the number of steps it takes — comparisons, swaps, or other discrete operations — as a function of input size, giving computer scientists a shared vocabulary for comparing one algorithm against another independently of hardware. Differences between algorithms are invisible on small inputs, since any algorithm finishes near-instantly on a handful of values; the differences emerge only when reasoning about what happens as the input size `n` grows toward a thousand, a million, or a billion values. Running time is therefore analyzed in broad strokes — via a graph of step count against problem size — rather than by tallying exact operation counts.

### 3.5.2 Step Counts for Linear Search, Optimized Linear Search, and Binary Search

Three algorithms already introduced supply concrete step counts to compare, in terms of the number of elements `n`:

- Linear search, worst case: `n` steps.
- Linear search, checking only every other locker: `n/2` steps — still proportional to `n`.
- Binary search: `log₂n` steps — the same halving logic behind the pairwise-counting algorithm of 3.2.3, where dividing a problem of size `n` in half repeatedly leaves a single element after `log₂n` divisions.

### 3.5.3 Big O: An Upper Bound on the Dominant Term

**Big O** `(O = order of)`notation states that an algorithm takes on the order of some number of steps, give or take — not an exact count. Two simplifications follow from this looseness:

- **Constant factors are dropped.** `O(n)` and `O(n/2)` are both written `O(n)`: both grow at a linear rate as `n` increases, so only the dominant term — the one that grows fastest — is kept.
- **The base of a logarithm is dropped.** `O(log₂n)` is written `O(log n)`: changing the base only rescales the curve by a constant factor, which Big O already discards.

> Computer scientists ignore lower-order terms — a divide-by-two, a logarithm's base — and look only at the term that dominates as `n` grows arbitrarily large, since that term decides how the algorithm actually scales.

#### 3.5.3.1 Visualizing Growth: Straight Lines vs. a Flattening Curve

Plotted with step count on the y-axis and problem size on the x-axis, the `O(n)` and `O(n/2)` curves are straight lines — the `O(n/2)` line at half the slope of `O(n)` — and they remain straight no matter how far the x-axis is zoomed out, from a handful of elements to a billion. The `O(log n)` curve, by contrast, flattens further the more it is zoomed out: doubling `n` adds only one more step to a binary search.

```text
steps
  |                                          O(n)
  |                                      ,·'
  |                                  ,·'      O(n/2)
  |                              ,·'      ,·'
  |                          ,·'      ,·'
  |                      ,·'      ,·'
  |                  ,·'      ,·'           O(log n)
  |              ,·'      ,·'        ____________
  |          ,·'      ,·'  _______/
  |      ,·' ,·' ____/
  |  ,·'_____/
  +---------------------------------------------------> problem size
```

### 3.5.4 A Cheat Sheet of Common Running Times

| Running time | Description |
|---|---|
| `O(1)` | Constant — a fixed number of steps (1, 2, 4, 10, ...) regardless of input size |
| `O(log n)` | Logarithmic |
| `O(n)` | Linear |
| `O(n log n)` | "`n` times `log n`" steps |
| `O(n²)` | Quadratic ("`n` squared") |

### 3.5.5 Big O Applied to Linear and Binary Search

Linear search is **O(n)**: in the worst case the target sits at the very end of the lockers, or is altogether absent, and every locker must be opened to confirm either outcome. Getting lucky — finding the target on an early locker — does not change this classification, since Big O is conventionally applied to the **worst case**, the scenario that bounds how badly an algorithm could perform on an unfavorable input.

Binary search is **O(log n)**, provided the data is already sorted: halving the remaining lockers on every comparison reaches the target, or exhausts the search space, in `log₂n` steps — the same divide-and-conquer logic as the Week 0 phone book and 3.2.3's pairwise counting. Binary search depends entirely on this precondition: applied to unsorted data, each less-than/greater-than comparison discards a half that may or may not actually contain the target, degrading the search into a sequence of arbitrary, often-wrong eliminations rather than a reliable narrowing.

#### 3.5.5.1 When Sorting First Pays Off (and When It Doesn't)

Because binary search requires sorted data, and sorting itself costs steps, which strategy wins depends on how many times the data will be searched. Searching a collection exactly once does not justify the cost of sorting it first — plain linear search is cheaper overall. Searching the same collection many times changes the calculation: the one-time cost of sorting is paid once and then amortized across every subsequent binary search, which ends up cheaper per search than repeating a linear scan each time.

## 3.6 Big Omega, Big Theta, and Asymptotic Notation

### 3.6.1 Big Omega: A Lower Bound

**Big Omega (Ω)** is the lower-bound counterpart to Big O: how few steps an algorithm might take in the best case, if everything goes as luckily as possible.

- Linear search is **Ω(1)**: the target could be behind the very first locker checked, regardless of how many lockers there are in total — a fixed number of steps unrelated to `n`.
- Binary search is also **Ω(1)**: the target could sit exactly at the middle locker checked on the very first comparison.

Both algorithms therefore share the same lower bound despite having different upper bounds (`O(n)` for linear search, `O(log n)` for binary search) — together, the two bounds sketch a range of possible performance: very fast in a lucky best case, but in the general case an algorithm is not lucky every time it runs, so its performance tends to sit closer to its upper bound than its lower one.

### 3.6.2 Big Theta: When Upper and Lower Bounds Coincide

**Big Theta (Θ)** is used when an algorithm's upper and lower bounds coincide — when Big O and Big Omega describe the exact same number of steps — collapsing two pieces of information into one. Neither linear search (`O(n)`, `Ω(1)`) nor binary search (`O(log n)`, `Ω(1)`) qualifies for Θ yet, since each has a best case strictly faster than its worst case; an algorithm earns a Θ classification only once no gap remains between how well and how poorly it can perform.

### 3.6.3 Asymptotic Notation, Defined

**Asymptotic notation** is the collective term for Big O, Big Omega, and Big Theta. *Asymptotic* describes a quantity that keeps growing as `n` grows arbitrarily large, without necessarily ever reaching a fixed boundary — the notation describes an algorithm's trajectory at that scale, not an exact step count for any one input size.

## 3.7 Implementing Linear Search in C

### 3.7.1 Searching an Array of Integers

A program, `search0.c`, implements linear search concretely:
```c
#include <cs50.h>
#include <stdio.h>

int main(void)
{
    int numbers[] = {20, 500, 10, 5, 100, 1, 50};
    int n = get_int("Number: ");
    for (int i = 0; i < 7; i++)
    {
        if (numbers[i] == n)
        {
            printf("Found\n");
            return 0;
        }
    }
    printf("Not found\n");
    return 1;
}
```
The array's size (7) is left for the compiler to infer from the number of values inside the curly braces, rather than stated explicitly in the square brackets — explicit sizing is permitted but risks a mismatch between the declared size and the actual element count. Returning `0` signals success and `1` signals failure, following the exit-status convention introduced in Week 2. Hard-coding the loop bound `7` is not ideal style but is acceptable here since the literal is not reused elsewhere in the program. Testing confirms the program finds `50` (the last element) and `20` (the first element), and correctly reports `1000` as not found.

### 3.7.2 Searching an Array of Strings: Why `==` Fails

Adapting the same program to search an array of strings (six Monopoly game pieces) surfaces a new bug:
```c
string strings[] = {"battleship", "boot", "cannon", "iron", "thimble", "top hat"};
string s = get_string("String: ");
for (int i = 0; i < 6; i++)
{
    if (strings[i] == s)
    {
        printf("Found\n");
        return 0;
    }
}
```
Running this reports `"battleship"`, `"boot"`, and `"top hat"` as **not found**, even though each is genuinely present in the array.

> Comparing strings with `==` does not work the way it does for integers. An integer comparison is a single, direct check; a string is a sequence of characters, and determining equality requires checking every character in sequence, which `==` does not do.

The fix is `strcmp`, from `string.h` (in the same family as the previously introduced `strlen`). `strcmp` walks both strings left to right, returning `0` if they are equal, a negative number if the first string would come before the second ("asciibetically" — based on underlying ASCII values, which only approximates true alphabetical order), and a positive number if it would come after:
```c
if (strcmp(strings[i], s) == 0)
```
A second bug appears on first attempting this fix: a compiler error reading roughly "call to undeclared library function `strcmp`" — caused by omitting `#include <string.h>`. Adding the header resolves it. The corrected program:
```c
#include <cs50.h>
#include <stdio.h>
#include <string.h>

int main(void)
{
    string strings[] = {"battleship", "boot", "cannon", "iron", "thimble", "top hat"};
    string s = get_string("String: ");
    for (int i = 0; i < 6; i++)
    {
        if (strcmp(strings[i], s) == 0)
        {
            printf("Found\n");
            return 0;
        }
    }
    printf("Not found\n");
    return 1;
}
```
> Comparing strings with `==` is a common bug: the code compiles and can appear to "almost work," but silently fails to detect equal strings. String equality in C always requires `strcmp(a, b) == 0`.

`strcmp`'s three-way return value (negative/zero/positive, not just true/false) exists because the same function is reused later for **sorting** strings — knowing which of two strings comes first, not merely whether they are equal, is what makes alphabetical ordering possible.

### 3.7.3 Hard-Coded Bounds and the Risk of a Segmentation Fault

Both `search0.c` and `search1.c` hard-code their loop bounds (`7` and `6`, respectively) rather than computing them from the array itself. This is flagged as poor style for two reasons: it requires the programmer to count the array's elements by hand, and it silently invites a specific class of bug if that count is ever miscopied.

A **segmentation fault** occurs when a program touches a part of memory it does not have permission to access — for instance, reading or writing past the last valid index of an array. If `search1.c`'s bound were mistakenly written as `i < 7` (carried over by analogy from `search0.c`'s seven-element array of integers) instead of `i < 6`, the loop would attempt to read `strings[6]` — one element past the end of a six-element array, in memory the array was never allocated to use. The call `strcmp(strings[6], s)` would then operate on unpredictable data, and the program would most likely crash with a segmentation fault rather than silently produce a wrong answer.

> Caution: a segmentation fault is the symptom, not the cause. The underlying cause is almost always an index that has drifted outside an array's valid range (`0` through `n-1`) — checking the loop bound against the array's actual size is the first thing to verify when one appears.

## 3.8 Arrays, Parallel Data, and the Case for `struct`

### 3.8.1 A Phone Book With Parallel Arrays

`phonebook0.c` reconstructs the Week 0 phone-book lookup using two parallel arrays:
```c
#include <cs50.h>
#include <stdio.h>
#include <string.h>

int main(void)
{
    string names[] = {"Kelly", "David", "John"};
    string numbers[] = {"+1-617-495-1000", "+1-617-495-1000", "+1-949-468-2750"};

    string name = get_string("Name: ");
    for (int i = 0; i < 3; i++)
    {
        if (strcmp(names[i], name) == 0)
        {
            printf("Found %s\n", numbers[i]);
            return 0;
        }
    }
    printf("Not found\n");
    return 1;
}
```
Both arrays are declared as `string`, even though phone numbers look numeric: numbers often contain punctuation (`+`, `-`) that an integer type cannot hold, and a leading zero in a phone number (common in regional dialing prefixes outside the US) would be silently dropped if stored as an integer, since leading zeros carry no mathematical value. Storing the number as a string of characters preserves it intact.

The search itself is a linear scan: names are not stored in sorted order, so binary search is not an option. Once `names[i]` matches the query, the corresponding `numbers[i]` is read — the two arrays stay aligned only by an "honor system," since nothing in the language enforces that they remain the same length and order.

> Parallel arrays place the burden of synchronization entirely on the programmer. Harmless for a handful of hard-coded entries, this becomes risky at scale — hundreds, millions of entries, or any database — since nothing stops the arrays from drifting out of sync.

### 3.8.2 Defining a Custom Type with `typedef struct`

C provides no built-in type that bundles a name and a number together; only primitives (`int`, `float`, `char`, `bool`, `string`) exist natively. The `typedef struct` keyword pair defines a new aggregate type combining several fields:
```c
typedef struct
{
    string name;
    string number;
} person;
```
This declares a new type, `person`, where every `person` has a `string name` and a `string number`. Placing this declaration above `main` makes it available to `main` and to any function defined later in the file. Unlike declaring separate variables for each field of each entry, `struct` groups *related but differently typed* data together — a role arrays alone cannot fill, since an array can only hold values of a single type.

### 3.8.3 A Phone Book With Structs

`phonebook1.c` replaces the two parallel arrays with a single array of `person`:
```c
#include <cs50.h>
#include <stdio.h>
#include <string.h>

typedef struct
{
    string name;
    string number;
} person;

int main(void)
{
    person people[3];

    people[0].name = "Kelly";
    people[0].number = "+1-617-495-1000";
    people[1].name = "David";
    people[1].number = "+1-617-495-1000";
    people[2].name = "John";
    people[2].number = "+1-949-468-2750";

    string name = get_string("Name: ");
    for (int i = 0; i < 3; i++)
    {
        if (strcmp(people[i].name, name) == 0)
        {
            printf("Found %s\n", people[i].number);
            return 0;
        }
    }
    printf("Not found\n");
    return 1;
}
```
Struct fields are accessed with dot notation: `people[i].name`, `people[i].number`. The search loop is otherwise unchanged — still 3 elements, each now holding two related values instead of one. This bundling of multiple fields into a single unit is called **encapsulation**: each element of `people` is a complete, self-contained record, and the related fields are also laid out contiguously in memory within each struct instance, rather than relying on convention to keep two independent arrays aligned.

## 3.9 Sorting

### 3.9.1 The Sorting Problem

Sorting, as a black box, takes unsorted input and produces sorted output: an input such as `7 2 5 4 1 6 0 3` should become `0 1 2 3 4 5 6 7`. Binary search's efficiency advantage over linear search depends entirely on the data having already been sorted, which motivates examining how sorting itself can be done, and at what cost.

### 3.9.2 Selection Sort

**Selection sort** repeatedly scans the unsorted portion of an array for its smallest remaining value and swaps that value into place at the front of that portion. In English:
```text
Repeat until no unsorted elements remain:
    Search the unsorted part of the data to find the smallest value
    Swap the smallest found value with the first element of the unsorted part
```
In array notation:
```text
For i from 0 to n-1
    Find smallest number between numbers[i] and numbers[n-1]
    Swap smallest number with numbers[i]
```
Traced on `7 2 5 4 1 6 0 3`: the smallest value overall (0) is found and swapped into index 0 (displacing the 7 that was there), giving `0 2 5 4 1 6 7 3`. The next pass searches only the remaining unsorted region (indices 1 through 7), finds 1, and swaps it into index 1, and so on — each pass fixes exactly one more element and never revisits an already-placed position.

The algorithm tracks only a single "smallest found so far" variable while scanning, rather than memorizing every value and position — a deliberate **trade-off between memory and time** that recurs throughout algorithm design. Because an array's size is fixed at the time of allocation, there is no spare slot to insert into; the smallest value found must instead be *swapped* with whatever currently occupies the destination index.

A complete run on the six-element array `5 2 1 3 6 4` (the same array used in 3.9.3.2's bubble-sort trace) shows the process through to completion, including the case where the smallest unsorted value is already the first unsorted element:

| Step | Unsorted part | Smallest found | First unsorted element | Action | Array after |
|---|---|---|---|---|---|
| 1 | 5,2,1,3,6,4 | 1 | 5 | swap 1 ↔ 5 | `1 2 5 3 6 4` |
| 2 | 2,5,3,6,4 | 2 | 2 | swap 2 ↔ 2 (no-op) | `1 2 5 3 6 4` |
| 3 | 5,3,6,4 | 3 | 5 | swap 3 ↔ 5 | `1 2 3 5 6 4` |
| 4 | 5,6,4 | 4 | 5 | swap 4 ↔ 5 | `1 2 3 4 6 5` |
| 5 | 6,5 | 5 | 6 | swap 5 ↔ 6 | `1 2 3 4 5 6` |
| 6 | 6 | 6 | — | only element left | `1 2 3 4 5 6` |

> When the smallest unsorted value happens to already sit at the front of the unsorted part (step 2 above), the "swap" is a no-op — the value is swapped with itself, leaving it in place but still confirmed sorted.

**Cost.** The first pass makes `n-1` comparisons, the second `n-2`, and so on down to a final pass of one comparison:
```text
(n-1) + (n-2) + (n-3) + ... + 1 = n(n-1)/2 = (n² - n)/2 = n²/2 - n/2
```
Keeping only the dominant term, selection sort is **O(n²)**. The pseudocode contains no check for whether the data is already sorted — every pass performs the same scan regardless of input — so the best case is also **Ω(n²)**, and since the upper and lower bounds coincide, selection sort is **Θ(n²)**: it always does the same amount of work, with no favorable case.

### 3.9.3 Bubble Sort

**Bubble sort** compares each pair of adjacent elements and swaps them if out of order, causing the largest unsorted value to "bubble" toward the end of the list one swap at a time over the course of a single pass. Where selection sort builds the sorted result left to right, smallest to largest, bubble sort builds it right to left, largest to smallest — each pass settles one more of the largest remaining values into place at the tail of the array, rather than the smallest value into place at its head.

Initial pseudocode:
```text
Repeat n times
    For i from 0 to n-2
        If numbers[i] and numbers[i+1] out of order
            Swap them
```
The inner loop stops at `n-2`, not `n-1`: since the comparison checks index `i` against `i+1`, allowing `i` to reach `n-1` (the last valid index) would push `i+1` to `n`, one past the end of the array.

> Off-by-one errors of exactly this kind — an index `i+1` overflowing past the last valid index — are a common source of bugs in array traversal.

Traced on `7 2 5 4 1 6 0 3`, one full pass moves the 7 rightward via successive adjacent swaps: `72541603` → `27541603` → `25741603` → `25471603` → `25417603` → `25416703` → `25416073` → `25416037`.

Continuing past that first pass, each subsequent pass bubbles the next-largest remaining value into place, one position further left of the previous one settled:

| Pass | Array after pass |
|---|---|
| 1 | `2 5 4 1 6 0 3 7` |
| 2 | `2 4 1 5 0 3 6 7` |
| 3 | `2 1 4 0 3 5 6 7` |
| 4 | `1 2 0 3 4 5 6 7` |
| 5 | `1 0 2 3 4 5 6 7` |
| 6 | `0 1 2 3 4 5 6 7` |
| 7 | `0 1 2 3 4 5 6 7` — no swaps occur; the array was already sorted after pass 6 |

The array reaches its fully sorted state after the sixth pass, yet the unoptimized pseudocode has no way of knowing that — it dutifully runs the seventh pass anyway, scanning all eight elements without making a single swap. This wasted final pass is exactly the inefficiency the early-exit optimization in 3.9.3.1 is designed to eliminate.


It suffices to repeat the outer loop only `n-1` times rather than `n`: once `n-1` elements have bubbled into place, the one remaining element is necessarily already correct.

**Cost.** Reading the cost directly from the pseudocode: the outer loop runs `n-1` times, the inner loop `n-1` times per outer iteration, and the comparison and swap are each constant-time regardless of list size. Treating the loops as nested (as with the row/column traversal used for printing a grid of characters), the total is:
```text
(n-1) × (n-1) = n² - 2n + 1
```
> Why classify this as **O(n²)** instead of just reporting the exact count from the trace above? Because that exact count — 7 passes × 7 comparisons per pass = 49 comparisons, per `(n-1)×(n-1)` — only describes *this* 8-element array. Recomputing a fresh exact count for every new array, and for every algorithm under consideration, wouldn't generalize or scale as a way to compare approaches. **O(n²)** instead answers a durable question: given a list twice as long, does the work roughly double, or roughly quadruple? That question has the same answer regardless of the specific 8 numbers traced above, which is what lets bubble sort be compared against selection sort (also O(n²)) and merge sort (O(n log n)) on one shared scale, without re-deriving either algorithm's exact step count from scratch.


Dropping lower-order terms, bubble sort is **O(n²)** — the same upper bound as selection sort.

#### 3.9.3.1 Early-Exit Optimization

Bubble sort can be enhanced: if a full pass makes no swaps, the list is already sorted, and the algorithm can quit immediately rather than running the remaining passes:
```text
Repeat n-1 times
    For i from 0 to n-2
        If numbers[i] and numbers[i+1] out of order
            Swap them
    If no swaps
        Quit
```
With this optimization, the best case requires only a single pass through an already-sorted list, giving bubble sort a lower bound of **Ω(n)**. Because the lower bound (Ω(n)) no longer matches the upper bound (O(n²)), bubble sort with the early-exit check can no longer be described with **Θ** — its average and worst cases remain on the order of n²; the optimization improves only the best case.

#### 3.9.3.2 Bubble Sort: Tracking Termination with a Swap Counter

The early-exit check from 3.9.3.1 ("if no swaps, quit") can be implemented concretely with a **swap counter**: a variable incremented once for every swap made during a pass, and reset to `0` at the start of each new pass.
```text
Set swap counter to a non-zero value
Repeat until the swap counter is 0:
    Reset swap counter to 0
    Look at each adjacent pair
    If two adjacent elements are not in order, swap them and add 1 to the swap counter
```
The counter is deliberately initialized to a nonzero placeholder (e.g., `-1`) before the loop begins — if it started at `0`, the "repeat until 0" condition would already be satisfied and the algorithm would never run a single pass. Once inside the loop, the counter is reset to `0` at the top of every pass and tracks only that pass's swaps; if a full pass completes with the counter still at `0`, no adjacent pair was out of order, and the array is provably sorted.

Traced on a six-element array `5 2 1 3 6 4`:

| Pass | Swaps made | Array after pass | Swap counter |
|---|---|---|---|
| 1 | (5,2), (5,1), (5,3), (6,4) | `2 1 3 5 4 6` | 4 |
| 2 | (2,1), (5,4) | `1 2 3 4 5 6` | 2 |
| 3 | none | `1 2 3 4 5 6` | 0 → stop |

The third pass, which makes no swaps, is what proves the array sorted and terminates the loop — the same mechanism as 3.9.3.1's "if no swaps, quit," made concrete with a counted variable rather than a boolean flag.

> The worst case (a fully reverse-sorted array) can also be understood spatially: every element may need to move across the *entire* array — large elements bubbling all the way to the right, small elements bubbling all the way to the left — so each of the `n` elements can require up to `n` steps of movement, reinforcing the `O(n²)` bound alongside the `(n-1) × (n-1)` algebraic derivation already given.

## 3.10 Recursion

### 3.10.1 Recursive Functions, Base Cases, and Recursive Cases

**Recursion** solves a problem by having a function call itself — a **recursive function** is one defined in terms of itself, mirroring a mathematical definition of the form `f(x) = f(something)`.

> A function calling itself appears to risk an infinite loop, since nothing obviously stops the calls from continuing indefinitely.

Two terms resolve this: a **base case** is a condition answerable immediately, with no further recursive work — it is what stops the recursion. A **recursive case** is the part of the function that calls itself again with a modified, strictly smaller input, moving the problem toward the base case. So long as every recursive call shrinks the problem, the recursion eventually reaches the base case and terminates; a call that handed off the same-sized problem would never make progress.

#### 3.10.1.1 The Factorial Function: A Worked Recursive/Iterative Comparison

The mathematical **factorial** function, conventionally written `n!`, multiplies together every positive integer from `n` down to `1` (e.g., `5! = 5 × 4 × 3 × 2 × 1`). Written as a function `fact(n)`, the pattern is suggestive: `fact(2) = 2 × fact(1)`, `fact(3) = 3 × fact(2)`, `fact(4) = 4 × fact(3)`, and so on — each value is `n` times the factorial of `n - 1`. Generalizing:
```text
fact(n) = n * fact(n-1)
```
This gives a base case (`fact(1) = 1`, which needs no further multiplication) and a recursive case (every other `n`, expressed in terms of a smaller call to the same function):
```c
int fact(int n)
{
    if (n == 1)
        return 1;
    else
        return n * fact(n-1);
}
```
The equivalent iterative version uses a loop and an accumulator variable instead of a self-call:
```c
int fact2(int n)
{
    int product = 1;
    while (n > 0)
    {
        product *= n;
        n--;
    }
    return product;
}
```
Both compute the same result; the recursive version expresses the *definition* of factorial almost directly in code, while the iterative version explicitly manages a running product and a countdown — the same trade-off already seen between `recursion.c` and `iteration.c`'s pyramid-drawing functions.

#### 3.10.1.2 Multiple Base Cases: The Fibonacci Sequence

A recursive function is not limited to a single base case or a single recursive case. The **Fibonacci sequence** — 0, 1, 1, 2, 3, 5, 8, ... — illustrates a function with two base cases: the first element is defined as `0` and the second as `1`, and every later element is the sum of the two preceding it.
```text
If n is 1, return 0.
If n is 2, return 1.
Otherwise, return fibonacci(n-1) + fibonacci(n-2).
```
Here, two separate conditions (`n == 1` and `n == 2`) each stop the recursion immediately, rather than just one.

#### 3.10.1.3 Multiple Recursive Cases: The Collatz Conjecture

A function can likewise have more than one recursive case. The **Collatz conjecture** speculates that, for any positive integer `n`, repeatedly applying a simple rule always eventually reaches `1`:
```text
If n is 1, stop.
Otherwise, if n is even, repeat this process on n / 2.
Otherwise, if n is odd, repeat this process on 3n + 1.
```
Counting how many steps this takes to reach `1` gives a single base case (`n == 1`, `0` steps) alongside *two* distinct recursive cases — one for even `n`, one for odd `n`:
```c
int collatz(int n)
{
    // base case
    if (n == 1)
        return 0;
    // even numbers
    else if ((n % 2) == 0)
        return 1 + collatz(n/2);
    // odd numbers
    else
        return 1 + collatz(3*n + 1);
}
```
Worked examples: `collatz(2)` takes 1 step (`2 → 1`); `collatz(3)` takes 7 steps (`3 → 10 → 5 → 16 → 8 → 4 → 2 → 1`); `collatz(7)` takes 16 steps (`7 → 22 → 11 → 34 → 17 → 52 → 26 → 13 → 40 → 20 → 10 → 5 → 16 → 8 → 4 → 2 → 1`).

> Despite no proof that the process always terminates for every positive integer, the conjecture has been verified across enormous ranges of numbers — illustrating that a recursive function's *correctness* (does it compute the right answer) and its *termination* (does it provably always reach a base case) are separate questions. The Collatz recursion above terminates for every input ever checked, without there existing a general proof that it always must.

### 3.10.2 Recursion Revisited: The Phone-Book Search

The Week 0 phone-book search, and the locker search earlier in this lecture, are reframed as the course's first (implicit) examples of recursion. The original procedural formulation used explicit "go back to" jumps to induce looping:
```text
1  Pick up phone book
2  Open to middle of phone book
3  Look at page
4  If person is on page
5      Call person
6  Else if person is earlier in book
7      Open to middle of left half of book
8      Go back to line 3
9  Else if person is later in book
10     Open to middle of right half of book
11     Go back to line 3
12 Else
13     Quit
```
This collapses into a recursive formulation by treating lines 7–8 ("open to the middle of the left half," then "go back to line 3") as a single instruction — "search the left half of the book," restating the whole algorithm on a smaller book — and likewise collapsing lines 10–11 into "search the right half of the book":
```text
1  Pick up phone book
2  Open to middle of phone book
3  Look at page
4  If person is on page
5      Call person
6  Else if person is earlier in book
7      Search left half of book
9  Else if person is later in book
10     Search right half of book
12 Else
13     Quit
```
> Line numbers 8 and 11 are intentionally absent here, not renumbered away — they mark exactly which steps each recursive call subsumes. "Search left half of book" *is* "open to the middle of the left half, look at the page, and (if necessary) go back to line 3," collapsed into one self-referential instruction, since the procedure being invoked is the very same search.

Lines 7 and 10 are now recursive calls — the same algorithm invoked again on a halved problem — with no explicit jump to a named line. The base case is the check at line 4 (found) or line 12/13 (none left, quit); the recursive case is searching the left or right half, each strictly smaller than the last.

### 3.10.3 A Recursive Definition: the Pyramid

A half-pyramid (the shape built in Problem Set 1) can itself be defined recursively: a pyramid of height 4 is a pyramid of height 3 plus one more row; height 3 is height 2 plus one more row; height 2 is height 1 plus one more row; and height 1 is a single brick, stated directly rather than in terms of a smaller pyramid — the **base case**. Every other height is the **recursive case**, defined in terms of the same structure, one row shorter.

### 3.10.4 Drawing a Pyramid Iteratively

`iteration.c` draws a pyramid using nested loops:
```c
#include <cs50.h>
#include <stdio.h>

void draw(int n);

int main(void)
{
    int height = get_int("Height: ");
    draw(height);
}

void draw(int n)
{
    for (int i = 0; i < n; i++)
    {
        for (int j = 0; j < i + 1; j++)
        {
            printf("#");
        }
        printf("\n");
    }
}
```
The inner bound is `i + 1`, not `i`: row 0 should print one brick, and since `i` starts at 0, the inner loop must run from `j = 0` to `j < 1` to produce exactly one `#` on that row, yielding 1, 2, 3, 4 bricks on successive rows.

> A first attempt at compiling failed because `draw` was called in `main` before being fully defined later in the file, with no function prototype declared at the top. Adding `void draw(int n);` above `main` resolves it: the compiler then knows `draw`'s signature before it is used, even though the full definition appears later in the file.

**Iteration** is defined plainly here as using loops to solve a problem.

### 3.10.5 Drawing a Pyramid Recursively

```c
// Line 1: Function Declaration
void draw(int n)
{
    // Line 2: The Base Case (The Eject Button / Safety Brake)
    if (n <= 0)
        return;

    // Line 3: The Recursive Call (The Pause Point)
    draw(n - 1);

    // Line 4: The Action (The Print Loop)
    for (int i = 0; i < n; i++)
        printf("#");
    printf("\n");
}
```

*Note on Syntax:* In C, if an `if` statement has only one line of code inside it, you can omit the curly braces `{ }`. That is why `if (n <= 0) return;` is written without braces.

---

### 1. The Base Case (Line 2)
Why do we check `if (n <= 0)` first?
* **The Safety Brake:** Without this check, the function would call itself forever. Modern compilers (like `clang`) will actually refuse to compile the program, warning you that *"all paths through this function will call itself."*
* **Why `<= 0` instead of `== 0`?** It is a defensive coding practice. If someone accidentally calls the function with a negative number (like `draw(-5)`), the program will immediately eject instead of breaking.

---

### 2. The Call Stack & Return Addresses (How the Computer Tracks it)
When a function calls itself, the computer doesn't just jump around randomly. It uses a **Call Stack** (think of a stack of sticky notes on your desk).

* **The Breadcrumbs (Return Address):** When `draw(3)` reaches Line 3, it calls `draw(2)`. Before jumping, the computer writes a note: *"I am in draw(3), paused at Line 3. Resume at Line 4 when I get back."* It puts this sticky note on the stack.
* **Stacking up:** The computer only works on the sticky note at the very top of the pile. As it calls `draw(2)`, `draw(1)`, and `draw(0)`, it piles up these notes.
* **The Eject (`return;`):** When it hits `n = 0`, Line 2 triggers the `return;`. This tells the computer to throw away the top sticky note (`draw(0)`) and look at the note right beneath it (`draw(1)`).
* **Conceptually "Deleting" lines 1-3:** When the computer goes back to `draw(1)`, it knows it already executed lines 1-3 for this copy. Those lines are "checked off" the to-do list. The computer skips them and goes straight to **Line 4** (the loop).

---

### 3. Step-by-Step Flow for `draw(3)`
Here is the exact line-by-line path the computer takes:

#### **Phase 1: Going Down (Calling & Stacking)**
1. **Line 1** (Starts `draw(3)`, where `n=3`)
2. **Line 2** (Check: `3 <= 0`? No.)
3. **Line 3** (Calls `draw(2)` $\rightarrow$ *Pauses `draw(3)`*)
4. **Line 1** (Starts `draw(2)`, where `n=2`)
5. **Line 2** (Check: `2 <= 0`? No.)
6. **Line 3** (Calls `draw(1)` $\rightarrow$ *Pauses `draw(2)`*)
7. **Line 1** (Starts `draw(1)`, where `n=1`)
8. **Line 2** (Check: `1 <= 0`? No.)
9. **Line 3** (Calls `draw(0)` $\rightarrow$ *Pauses `draw(1)`*)
10. **Line 1** (Starts `draw(0)`, where `n=0`)
11. **Line 2** (Check: `0 <= 0`? Yes $\rightarrow$ `return;` ejects `draw(0)`)

#### **Phase 2: Going Up (Resuming & Printing)**
*(The computer returns to the top of the stack and runs the remaining Line 4)*

12. **Line 4** (Resumes `draw(1)`, prints `#`) $\rightarrow$ `draw(1)` finishes and ejects.
13. **Line 4** (Resumes `draw(2)`, prints `##`) $\rightarrow$ `draw(2)` finishes and ejects.
14. **Line 4** (Resumes `draw(3)`, prints `###`) $\rightarrow$ `draw(3)` finishes and exits.

---

### 4. Scope: Clean Slates for Variables
Each copy of the function gets its own isolated memory block. 
* The loop variable `i` inside `draw(3)` is completely separate and invisible to `draw(2)` or `draw(1)`. 
* They do not share or overwrite each other's variables.

---

### 5. Memory Caution (Stack Overflow)
Unlike a normal `for` loop, recursion uses extra memory for every single call because it has to keep storing those sticky notes (Stack Frames) in memory.
* If you input a massive height (like `draw(1000000)`), the pile of sticky notes gets too tall and spills out of the allowed memory.
* The computer runs out of space, and the Operating System will crash your program (known as a **Stack Overflow** or **Segmentation Fault**).

## 3.11 Merge Sort

### 3.11.1 Pseudocode and the Merge Step

**Merge sort**, a third sorting algorithm, uses recursion to make far fewer comparisons than selection or bubble sort, avoiding repeated work on the same elements.

Pseudocode:
```text
Sort the left half of the numbers
Sort the right half of the numbers
Merge the sorted halves
```
This is self-referential — the first two lines invoke the same procedure on smaller inputs — making it recursive by construction. The base case: a list of size 0 or 1 is already sorted and requires no further work.

**Merging** two already-sorted halves of equal size proceeds by comparing the foremost unconsumed element of each half, taking the smaller of the two, and advancing only the pointer belonging to the half that contributed it. Given a left half `1, 3, 4, 6` and a right half `0, 2, 5, 7`:

| Compare | Take | Advance |
|---|---|---|
| 1 vs 0 | 0 | right |
| 1 vs 2 | 1 | left |
| 3 vs 2 | 2 | right |
| 3 vs 5 | 3 | left |
| 4 vs 5 | 4 | left |
| 6 vs 5 | 5 | right |
| 6 vs 7 | 6 | left (left exhausted) |
| — | 7 | remaining element |

Result: `0, 1, 2, 3, 4, 5, 6, 7`. Unlike selection or bubble sort, the merge pointers move only forward, touching each element exactly once; merging two sorted halves totaling n elements takes n steps.

### 3.11.2 Tracing Merge Sort on an 8-Element List

Starting with the unsorted list `[6, 3, 4, 1, 5, 2, 0, 7]`, merge sort uses recursion to divide the list before sorting and merging it back together.

---

#### 1. The Analogy: Sorting Paper Piles
Think of merge sort as sorting a messy stack of 8 test scores:
1. You keep splitting the stack in half until you have 8 individual stacks of **1 paper each** (each single paper is sorted by default). This is the **Base Case**.
2. You then merge the stacks back together, two at a time, sorting them as you combine them.

---

#### 2. Phase 1: Going Down (The Splitting)
The list is divided in half recursively. The left side is completely split to its base cases before the right side is processed:

1. **Start:** `[6, 3, 4, 1, 5, 2, 0, 7]`
2. **Split 1 (Left Half):** `[6, 3, 4, 1]`
   * **Split 2 (Left-Left):** `[6, 3]`
     * **Split 3 (Base Cases):** `[6]` and `[3]` (Cannot split further; begins merging).
   * **Split 4 (Left-Right):** `[4, 1]`
     * **Split 5 (Base Cases):** `[4]` and `[1]` (Cannot split further; begins merging).
3. **Split 6 (Right Half):** `[5, 2, 0, 7]`
   * **Split 7 (Right-Left):** `[5, 2]`
     * **Split 8 (Base Cases):** `[5]` and `[2]` (Cannot split further).
   * **Split 9 (Right-Right):** `[0, 7]`
     * **Split 10 (Base Cases):** `[0]` and `[7]` (Cannot split further).

---

#### 3. Phase 2: Going Up (The Merging & Sorting)
Once the base cases are reached, the computer "unwinds" the recursion, merging the sub-lists back together in sorted order:

1. **Merge Level 1 (Individual elements into pairs):**
   * Merge `[6]` and `[3]` $\rightarrow$ `[3, 6]`
   * Merge `[4]` and `[1]` $\rightarrow$ `[1, 4]`
   * Merge `[5]` and `[2]` $\rightarrow$ `[2, 5]`
   * Merge `[0]` and `[7]` $\rightarrow$ `[0, 7]`
2. **Merge Level 2 (2-element lists into 4-element lists):**
   * Merge `[3, 6]` and `[1, 4]` $\rightarrow$ `[1, 3, 4, 6]`
   * Merge `[2, 5]` and `[0, 7]` $\rightarrow$ `[0, 2, 5, 7]`
3. **Merge Level 3 (4-element lists into final 8-element list):**
   * Merge `[1, 3, 4, 6]` and `[0, 2, 5, 7]` $\rightarrow$ `[0, 1, 2, 3, 4, 5, 6, 7]`

---

#### 4. Visual Execution Diagram

```text
              [6, 3, 4, 1, 5, 2, 0, 7]             <- Unsorted Start
                    /          \
            [6, 3, 4, 1]      [5, 2, 0, 7]         <- Split 1
             /        \        /        \
          [6, 3]    [4, 1]   [5, 2]    [0, 7]      <- Split 2
          /    \    /    \   /    \    /    \
        [6]   [3]  [4]   [1][5]   [2] [0]   [7]    <- Base Cases (Size 1)
        =======================================
          \    /    \    /   \    /    \    /
          [3, 6]    [1, 4]   [2, 5]    [0, 7]      <- Merge Level 1
             \        /        \        /
            [1, 3, 4, 6]      [0, 2, 5, 7]         <- Merge Level 2
                    \          /
              [0, 1, 2, 3, 4, 5, 6, 7]             <- Final Sorted List

#### 3.11.2.1 Tracing Merge Sort on a Six-Element List (Odd Split)

The 8-element trace above always splits evenly, since 8, 4, and 2 are all even. A six-element array, `5, 2, 1, 3, 6, 4` — the same array used in 3.9.2's selection-sort trace and 3.9.3.2's bubble-sort trace — produces a sub-array of odd length partway through, illustrating how merge sort handles a split that can't be even.

When a sub-array has an odd number of elements, the convention is to make the **left half the smaller of the two** (e.g., splitting 3 elements into a 1-element left half and a 2-element right half). The specific choice is arbitrary; what matters is applying it **consistently** throughout the recursion, since merge sort's correctness doesn't depend on how a tie in size is broken, only on always breaking it the same way.

Tracing `5, 2, 1, 3, 6, 4`:

1. **Sort the left half** (`5, 2, 1`): split unevenly into left `5` (a base case) and right `2, 1` (split further into `2` and `1`, merged into `1, 2`). Merging `5` and `1, 2`: compare `5` vs `1` → take `1`; compare `5` vs `2` → take `2`; nothing remains on the right, so `5` is taken last. Result: `1, 2, 5`.
2. **Sort the right half** (`3, 6, 4`): split unevenly into left `3` (a base case) and right `6, 4` (split into `6` and `4`, merged into `4, 6`). Merging `3` and `4, 6`: compare `3` vs `4` → take `3`; nothing remains on the left, so the entire remaining right side (`4, 6`) is taken as-is. Result: `3, 4, 6`.
3. **Merge the two sorted halves**, `1, 2, 5` and `3, 4, 6`, comparing element by element (`1` vs `3` → `1`; `2` vs `3` → `2`; `5` vs `3` → `3`; `5` vs `4` → `4`; `5` vs `6` → `5`; `6` vs nothing → `6`) into the final sorted list `1, 2, 3, 4, 5, 6`.

> When one side of a merge is exhausted, the comparison becomes "the remaining value vs. nothing" — the remaining side's values are necessarily smaller (or larger) than everything already merged, since both sides arrived at the merge step already sorted, so they are taken in order without further comparison.

### 3.11.3 Big-O Analysis: Θ(n log n)

For `n = 8`, the algorithm operates at 3 distinct levels, and the total merging work across all sublists at any one level is `n` steps. The number of levels relates to `n` logarithmically: `8 = 2³`, so `log₂8 = 3`; in general, an input of size n splits into `log₂n` (written `log n`, base omitted under Big-O) levels. With `n` steps of work at each of `log n` levels, the total running time is on the order of **n log n**.

| Bound | Value |
|---|---|
| Big O (upper bound) | O(n log n) |
| Big Ω (lower bound) | Ω(n log n) |
| Big Θ | Θ(n log n) |

> Merge sort has no best-case shortcut analogous to bubble sort's early exit, so its lower bound matches its upper bound in all cases — even an already-sorted array must still be fully split and recombined, since the algorithm has no way to detect sortedness in advance. This is a strictly better asymptotic running time than either selection sort or bubble sort (both O(n²)), at the cost of roughly twice the memory — an extra buffer is needed to hold elements during each merge — a classic **time/space trade-off**.

### 3.11.4 Comparing All Three Sorting Algorithms

A side-by-side visualization sorting the same randomized data with all three algorithms running at the same speed shows merge sort finishing visibly faster than bubble sort or selection sort, since dividing the problem recursively touches far fewer total elements than repeatedly rescanning the whole list.

| Algorithm | Big O | Big Ω | Big Θ | Improvement #1 |
|---|---|---|---|---|
| Selection sort | O(n²) | Ω(n²) | Θ(n²) | — |
| Bubble sort | O(n²) | Ω(n) | — | Early exit when a pass makes no swaps |
| Merge sort | O(n log n) | Ω(n log n) | Θ(n log n) | — |